# 09 RAG Query and Grounding\n
Run retrieval and generation loop with source citation output.

In [ ]:
import json
import numpy as np
import faiss
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

import warnings
warnings.filterwarnings('ignore')

print("✓ Imports successful")

In [ ]:
# Test queries
test_queries = [
    "What is the relationship between debt and profitability?",
    "How do you calculate operating margin?",
    "What does EBITDA measure in financial analysis?"
]

results = []
for query in test_queries:
    result = rag_pipeline(query, k=2)
    results.append(result)

print(f"\n{'='*60}")
print(f"RAG Pipeline Test Complete!")
print(f"{'='*60}")
print(f"Processed {len(results)} queries successfully")

## 4. Test RAG Pipeline

In [ ]:
def retrieve_documents(query, k=3):
    """Retrieve top-k documents from index"""
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding.astype('float32'), k)
    
    retrieved_docs = []
    for idx in indices[0]:
        retrieved_docs.append({
            'doc_id': idx,
            'text': corpus_docs[idx],
            'metadata': corpus_metadata[idx]
        })
    
    return retrieved_docs

def generate_response(query, retrieved_docs):
    """Generate response using retrieved context"""
    # Build context from retrieved documents
    context = "\n".join([doc['text'] for doc in retrieved_docs])
    
    # Create prompt
    prompt = f"""[INST] Answer the financial question based on the provided context.

Context: {context}

Question: {query} [/INST]

Answer:"""
    
    # Generate
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            top_p=0.9,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the answer part
    answer_start = response.find("Answer:") + len("Answer:")
    answer = response[answer_start:].strip() if answer_start > len("Answer:") else response
    
    return answer, context

def rag_pipeline(query, k=3):
    """Full RAG pipeline: retrieve -> generate -> ground"""
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    
    # Step 1: Retrieve
    print(f"\n1️⃣  RETRIEVING {k} relevant documents...")
    retrieved_docs = retrieve_documents(query, k=k)
    
    for i, doc in enumerate(retrieved_docs, 1):
        print(f"  {i}. Doc ID: {doc['doc_id']}")
        print(f"     Text (first 150 chars): {doc['text'][:150]}...")
    
    # Step 2: Generate
    print(f"\n2️⃣  GENERATING response with retrieved context...")
    answer, context = generate_response(query, retrieved_docs)
    
    # Step 3: Ground (show sources)
    print(f"\n3️⃣  GROUNDED RESPONSE:")
    print(f"  {answer}")
    
    print(f"\n4️⃣  SOURCES CITED:")
    for i, doc in enumerate(retrieved_docs, 1):
        if 'question' in doc['metadata']:
            print(f"  [{i}] Related Q&A: {doc['metadata']['question'][:80]}...")
    
    return {
        'query': query,
        'answer': answer,
        'retrieved_docs': [d['doc_id'] for d in retrieved_docs],
        'context_used': context
    }

print("✓ RAG pipeline defined")

## 3. Define RAG Pipeline

In [ ]:
print("="*60)
print("Loading RAG Components")
print("="*60)

# Load FAISS index
corpus_path = Path("../data/corpus")
index_file = corpus_path / "faiss_index" / "financial_corpus.index"
metadata_file = corpus_path / "metadata.jsonl"

if not index_file.exists():
    print(f"⚠ Index not found. Run 08_rag_index_build.ipynb first")
else:
    print(f"\nLoading FAISS index...")
    index = faiss.read_index(str(index_file))
    print(f"✓ Loaded index with {index.ntotal} documents")
    
    # Load metadata
    corpus_metadata = []
    corpus_docs = []
    with open(metadata_file, 'r') as f:
        for line in f:
            meta = json.loads(line)
            corpus_metadata.append(meta)
            
    # Load corpus text from processed data
    with open(Path("../data/processed/train.jsonl"), 'r') as f:
        for line in f:
            sample = json.loads(line)
            corpus_docs.append(sample['context'])
    
    print(f"✓ Loaded {len(corpus_docs)} documents")
    
    # Load embedding model
    print(f"Loading embedding model...")
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    print(f"✓ Embedding model loaded")
    
    # Load fine-tuned model
    print(f"\nLoading fine-tuned LLM...")
    BASE_MODEL = "mistralai/Mistral-7B"
    ADAPTER_PATH = "../models/finance-adapter-pilot"
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    
    if Path(ADAPTER_PATH).exists():
        model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    
    model.eval()
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    print(f"✓ LLM loaded")

## 2. Load RAG Components